# Gene translation for external datasets

In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Load databases

In [2]:
ensdb <- EnsDb.Hsapiens.v86::EnsDb.Hsapiens.v86
orgdb <- org.Hs.eg.db::org.Hs.eg.db

# Cantoni et al. (2025)

### Load original genes

Original genes TSV generated by extracting feature names from the original counts matrix, converting to a data frame with a single column named 'gene_symbol' and identical row names/indices

In [ ]:
genes <- read.table("../data/processed/external/cantoni/source/cantoni_genes_original.tsv", sep = "\t", header = TRUE, row.names = 1) %>%
  dplyr::rename(original_gene_symbol = gene_symbol)

### Extract Ensembl ID genes

In [11]:
names <- sub(pattern = "[.][0-9]*", replacement = "", x = rownames(genes)) # remove version numbers
ensembl.genes <- grepl(pattern = "^ENSG[0-9]+", x = names)

### Process Ensembl ID genes

In [12]:
ensembl.genes <- genes[ensembl.genes, , drop = FALSE]

Map Ensembl IDs to canonical gene symbols from Ensembl database

In [13]:
ensembl.mapping <- ensembldb::select(
  ensdb,
  keys = rownames(ensembl.genes),
  keytype = "GENEID",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  ) %>%
  mutate(original_gene_symbol = ensembl_id)

Count unmapped genes

In [14]:
ensembl.unmapped <- setdiff(rownames(ensembl.genes), ensembl.mapping$original_gene_symbol)
length(ensembl.unmapped)

[1] 2940

### Process symbol genes

In [15]:
symbol.genes <- genes[setdiff(rownames(genes), rownames(ensembl.genes)), , drop = FALSE]

Map gene symbols to canonical gene symbols and extract Ensembl IDs from Ensembl database

In [16]:
symbol.mapping <- ensembldb::select(
  ensdb,
  keys = rownames(symbol.genes),
  keytype = "SYMBOL",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  mutate(original_gene_symbol = gene_symbol)

In [17]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)

Attempt to find aliases for unmapped gene symbols using HGNC database

In [18]:
aliases <- AnnotationDbi::select(
  orgdb,
  keys = symbol.unmapped,
  keytype = "ALIAS",
  columns = c("ENSEMBL", "ALIAS")
) %>%
  dplyr::rename(
    ensembl_id = ENSEMBL,
    original_gene_symbol = ALIAS
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(original_gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup()
aliases.mapping <- ensembldb::select(
  ensdb,
  keys = aliases$ensembl_id,
  keytype = "GENEID",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  )
aliases.mapping <- left_join(aliases, aliases.mapping, by = "ensembl_id")

'select()' returned 1:many mapping between keys and columns



In [19]:
symbol.mapping <- bind_rows(symbol.mapping, aliases.mapping) %>% distinct()

Count unmapped genes

In [20]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)
length(symbol.unmapped)

[1] 10784

### Combine results

In [21]:
mapping <- bind_rows(ensembl.mapping, symbol.mapping) %>% distinct()

Filter to only keep successfully mapped genes of selected types

In [23]:
df <- left_join(genes, mapping, by = "original_gene_symbol") %>%
  dplyr::filter(
    !is.na(gene_symbol),
    !is.na(ensembl_id),
    grepl(pattern = "protein_coding|lincRNA|IG_C_gene|TR_C_gene", x = gene_type)
  )

In [ ]:
# Save results
write.table(df, file = "../data/processed/external/cantoni/source/cantoni_genes_translation.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

# Absinta et al. (2021) and Lerma-Martin et al. (2024)

### Load original genes

Original genes TSV generated by extracting feature names from the original counts matrix, converting to a data frame with a single column named 'gene_symbol' and identical row names/indices

In [ ]:
genes <- read.table("../data/processed/external/lesion_rims/source/lesion_rims_genes_original.tsv", sep = "\t", header = TRUE, row.names = 1) %>%
  dplyr::rename(original_gene_symbol = gene_symbol)

### Extract Ensembl ID genes

In [ ]:
names <- sub(pattern = "[.][0-9]*", replacement = "", x = rownames(genes)) # remove version numbers
ensembl.genes <- grepl(pattern = "^ENSG[0-9]+", x = names)

In [ ]:
ensembl.genes <- genes[ensembl.genes, , drop = FALSE]

### Process symbol genes

In [ ]:
symbol.genes <- genes[setdiff(rownames(genes), rownames(ensembl.genes)), , drop = FALSE]

Map gene symbols to canonical gene symbols and extract Ensembl IDs from Ensembl database

In [ ]:
symbol.mapping <- ensembldb::select(
  ensdb,
  keys = rownames(symbol.genes),
  keytype = "SYMBOL",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  mutate(original_gene_symbol = gene_symbol)

In [ ]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)

Attempt to find aliases for unmapped gene symbols using HGNC database

In [ ]:
aliases <- AnnotationDbi::select(
  orgdb,
  keys = symbol.unmapped,
  keytype = "ALIAS",
  columns = c("ENSEMBL", "ALIAS")
) %>%
  dplyr::rename(
    ensembl_id = ENSEMBL,
    original_gene_symbol = ALIAS
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(original_gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup()
aliases.mapping <- ensembldb::select(
  ensdb,
  keys = aliases$ensembl_id,
  keytype = "GENEID",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  )
aliases.mapping <- left_join(aliases, aliases.mapping, by = "ensembl_id")

'select()' returned 1:many mapping between keys and columns



In [ ]:
symbol.mapping <- bind_rows(symbol.mapping, aliases.mapping) %>% distinct()

Count unmapped genes

In [ ]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)
length(symbol.unmapped)

[1] 11948

### Combine results

In [ ]:
mapping <- symbol.mapping

Filter to keep only successfully mapped genes of selected types

In [ ]:
df <- left_join(genes, mapping, by = "original_gene_symbol") %>%
  dplyr::filter(
    !is.na(gene_symbol),
    !is.na(ensembl_id),
    grepl(pattern = "protein_coding|lincRNA|IG_C_gene|TR_C_gene", x = gene_type)
  )

In [ ]:
# Save results
write.table(df, file = "../data/processed/external/lesion_rims/source/lesion_rims_genes_translation.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

# Kaufmann et al. (2021)

### Load original genes

Original genes TSV generated by extracting feature names from the original counts matrix, converting to a data frame with a single column named 'gene_symbol' and identical row names/indices

In [ ]:
genes <- read.table("../data/processed/external/kaufmann/source/kaufmann_genes_original.tsv", sep = "\t", header = TRUE, row.names = 1) %>%
  dplyr::rename(original_gene_symbol = gene_symbol)

### Extract Ensembl ID genes

In [4]:
names <- sub(pattern = "[.][0-9]*", replacement = "", x = rownames(genes)) # remove version numbers
ensembl.genes <- grepl(pattern = "^ENSG[0-9]+", x = names)

In [7]:
ensembl.genes <- genes[ensembl.genes, , drop = FALSE]

### Process symbol genes

In [8]:
symbol.genes <- genes[setdiff(rownames(genes), rownames(ensembl.genes)), , drop = FALSE]

Map gene symbols to canonical gene symbols and extract Ensembl IDs from Ensembl database

In [9]:
symbol.mapping <- ensembldb::select(
  ensdb,
  keys = rownames(symbol.genes),
  keytype = "SYMBOL",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup() %>%
  mutate(original_gene_symbol = gene_symbol)

In [10]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)

Attempt to find aliases for unmapped gene symbols using HGNC database

In [11]:
aliases <- AnnotationDbi::select(
  orgdb,
  keys = symbol.unmapped,
  keytype = "ALIAS",
  columns = c("ENSEMBL", "ALIAS")
) %>%
  dplyr::rename(
    ensembl_id = ENSEMBL,
    original_gene_symbol = ALIAS
  ) %>%
  # Remove NA mappings
  dplyr::filter(!is.na(ensembl_id)) %>%
  # Handle one-to-many mappings by keeping only the first Ensembl ID for each gene symbol
  # This prioritizes the primary/canonical gene entry when multiple IDs exist
  group_by(original_gene_symbol) %>%
  slice_head(n = 1) %>%
  ungroup()
aliases.mapping <- ensembldb::select(
  ensdb,
  keys = aliases$ensembl_id,
  keytype = "GENEID",
  columns = c("GENEID", "SYMBOL", "GENEBIOTYPE")
) %>%
  dplyr::rename(
    ensembl_id = GENEID,
    gene_symbol = SYMBOL,
    gene_type = GENEBIOTYPE
  )
aliases.mapping <- left_join(aliases, aliases.mapping, by = "ensembl_id")

'select()' returned 1:many mapping between keys and columns



In [12]:
symbol.mapping <- bind_rows(symbol.mapping, aliases.mapping) %>% distinct()

Count unmapped genes

In [13]:
symbol.unmapped <- setdiff(rownames(symbol.genes), symbol.mapping$original_gene_symbol)
length(symbol.unmapped)

[1] 97

### Combine results

In [15]:
mapping <- symbol.mapping

Filter to keep only successfully mapped genes of selected types

In [16]:
df <- left_join(genes, mapping, by = "original_gene_symbol") %>%
  dplyr::filter(
    !is.na(gene_symbol),
    !is.na(ensembl_id),
    grepl(pattern = "protein_coding|lincRNA|IG_C_gene|TR_C_gene", x = gene_type)
  )

In [ ]:
# Save results
write.table(df, file = "../data/processed/external/kaufmann/source/kaufmann_genes_translation.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

In [3]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.3 forcats_1.0.0   stringr_1.5.1   dplyr_1.1.4    
 [5] purrr_1.0.2     readr_2.1.5     tidyr_1.3.1     tibble_3.2.1   
 [9] ggplot2_3.5.1   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] DBI_1.2.3                   bitops_1.0-9               
 [3] biomaRt_2.58.0              rlang_1.1.4                
 [5] magrittr_2.0.3              matrixStats_1.4.1          
 [7] compiler_4.3.3              RSQLite_2.3.7              
 [9] GenomicFeatures_1.54.1      png_0.1-8                  
[11] vctrs_0.6.5                